# Data Mining Assignment: Apache Spark Data Lake (Bronze → Silver)

**Dataset:** `sales.csv` (e-commerce sales orders, 13 columns)
**Environment:** Google Colab + PySpark

This notebook:

1. Builds a simple data lake: `data_lake/bronze/` (raw, unchanged) and `data_lake/silver/` (preprocessed).
2. Loads the Bronze file with Spark and inspects it: schema, sample rows, record count.
3. Uses Spark to find **duplicates, missing values, text-format inconsistencies, invalid numbers and inconsistent or invalid dates**.
4. Applies basic preprocessing and saves the result to the **Silver** layer.
5. Verifies the output by reading the Silver data back with Spark.

> Every discrepancy count in this notebook comes from a Spark operation. None are hard-coded.
> Run it top to bottom with **Runtime → Run all**.

## 1. Environment setup

Install PySpark. Spark 3.5 runs on Java 8, 11 or 17. On Colab the notebook installs OpenJDK 17 if it is missing, so the Java version is predictable.

In [1]:
import os, sys, shutil

IN_COLAB = "google.colab" in sys.modules

%pip install -q pyspark==3.5.9

if IN_COLAB and not os.path.isdir("/usr/lib/jvm/java-17-openjdk-amd64"):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
if IN_COLAB:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("Running in Google Colab:", IN_COLAB)
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(system default)"))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.9 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Running in Google Colab: True
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


## 2. Data lake structure

```
data_lake/
|-- bronze/
|   `-- sales.csv              <- raw file, byte-for-byte copy, read-only
`-- silver/
    |-- preprocessed_sales/    <- preprocessed data written by Spark (Parquet)
    `-- sales_silver.csv       <- the same Silver data as one CSV file (for submission)
```

The raw file goes into Bronze **exactly as provided**. If `sales.csv` was uploaded to the Colab session (Files panel → Upload), the notebook copies it. If not, it downloads the file from the class repository. A SHA-256 checksum confirms the Bronze file is identical to the source, and the file is set to read-only.

In [2]:
import hashlib, urllib.request
from pathlib import Path

LAKE         = Path("data_lake")
BRONZE_DIR   = LAKE / "bronze"
SILVER_DIR   = LAKE / "silver"
BRONZE_FILE  = BRONZE_DIR / "sales.csv"
SILVER_TABLE = SILVER_DIR / "preprocessed_sales"   # Spark output folder (Parquet)
SILVER_CSV   = SILVER_DIR / "sales_silver.csv"     # single-file CSV copy of the Silver data

SOURCE_URL = ("https://raw.githubusercontent.com/necromancersanju-netizen/"
              "data-mining-data-lake-assignment/main/dataset/sales.csv")

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_TABLE.mkdir(parents=True, exist_ok=True)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def show_tree(root):
    root = Path(root)
    print(root.name + "/")
    def walk(folder, prefix):
        entries = sorted((e for e in folder.iterdir() if not e.name.endswith(".crc")),
                         key=lambda e: (e.is_file(), e.name))
        for i, e in enumerate(entries):
            last = i == len(entries) - 1
            print(prefix + ("`-- " if last else "|-- ") + e.name + ("/" if e.is_dir() else ""))
            if e.is_dir():
                walk(e, prefix + ("    " if last else "|   "))
    walk(root, "")

# Land the raw file in Bronze once. It is never edited afterwards.
if not BRONZE_FILE.exists():
    if Path("sales.csv").exists():                    # uploaded to the session
        shutil.copyfile("sales.csv", BRONZE_FILE)
        print("Bronze file copied from uploaded sales.csv")
    else:                                             # fetch from the class repository
        urllib.request.urlretrieve(SOURCE_URL, BRONZE_FILE)
        print("Bronze file downloaded from", SOURCE_URL)
    os.chmod(BRONZE_FILE, 0o444)                      # read-only: the Bronze layer is immutable

BRONZE_SHA256 = sha256(BRONZE_FILE)
print("Bronze file :", BRONZE_FILE, f"({BRONZE_FILE.stat().st_size:,} bytes)")
print("SHA-256     :", BRONZE_SHA256)
if Path("sales.csv").exists():
    print("Identical to the provided sales.csv:", BRONZE_SHA256 == sha256("sales.csv"))
print()
show_tree(LAKE)

Bronze file downloaded from https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv
Bronze file : data_lake/bronze/sales.csv (109,537 bytes)
SHA-256     : 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8

data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/


## 3. Create the SparkSession

`spark.sql.legacy.timeParserPolicy = CORRECTED` makes an impossible date such as `2025-02-30` parse to `NULL` instead of raising an error. That way invalid dates can be found and handled.

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("Sales-DataLake-Bronze-to-Silver")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 3.5.9


## 4. Load and inspect the Bronze data

### 4.1 Schema, sample rows and record count

In [4]:
bronze_df = spark.read.csv(str(BRONZE_FILE), header=True, inferSchema=True)

bronze_df.printSchema()
bronze_df.show(5, truncate=False)
print("Total records in Bronze:", bronze_df.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

`order_date` is inferred as a **string**, which already suggests mixed or invalid date values.

For discrepancy detection the file is read again with **every column as a string**. This way no value is silently converted or lost before it is inspected. The helper functions below are the data-quality checks. They are reused in Section 7 to check the Silver output.

In [5]:
raw_df = spark.read.csv(str(BRONZE_FILE), header=True, inferSchema=False)   # all columns as strings
raw_df.cache()

COLS      = raw_df.columns
TEXT_COLS = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
NUM_COLS  = ["quantity", "unit_price", "discount_percent"]
TOTAL_ROWS = raw_df.count()

def num(c):
    return F.col(c).cast("double")

# A value is invalid if it is present but non-numeric or outside its allowed range
INVALID = {
    "quantity":         F.col("quantity").isNotNull()
                        & (num("quantity").isNull() | (num("quantity") <= 0) | (num("quantity") != F.floor(num("quantity")))),
    "unit_price":       F.col("unit_price").isNotNull()
                        & (num("unit_price").isNull() | (num("unit_price") <= 0)),
    "discount_percent": F.col("discount_percent").isNotNull()
                        & (num("discount_percent").isNull() | (num("discount_percent") < 0) | (num("discount_percent") > 100)),
}
RULES = {"quantity": "whole number >= 1", "unit_price": "number > 0", "discount_percent": "number between 0 and 100"}

# Date formats seen in the data: ISO, year-first with slashes, day-first with slashes, month-first with dashes
DATE_FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "MM-dd-yyyy"]
PARSED_DATE  = F.coalesce(*[F.to_date(F.trim(F.col("order_date")), f) for f in DATE_FORMATS])
DATE_STATUS = (F.when(F.col("order_date").isNull(), "missing")
                .when(PARSED_DATE.isNull() & F.trim(F.col("order_date")).rlike(r"^\d{1,4}[-/]\d{1,2}[-/]\d{1,4}$"),
                      "invalid calendar date")
                .when(PARSED_DATE.isNull(), "not a date")
                .when(PARSED_DATE > F.current_date(), "future date")
                .when(F.trim(F.col("order_date")).rlike(r"^\d{4}-\d{2}-\d{2}$"), "valid, yyyy-MM-dd")
                .otherwise("valid, non-standard format"))

def missing_counts(df, cols):
    row = df.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
                     for c in cols]).first()
    return {c: int(row[c] or 0) for c in cols}

def whitespace_counts(df, cols):
    row = df.select([F.sum(F.when(F.col(c) != F.trim(F.col(c)), 1).otherwise(0)).alias(c) for c in cols]).first()
    return {c: int(row[c] or 0) for c in cols}

def case_variant_rows(df, c):
    # Rows whose (trimmed) spelling differs from the most common spelling of the same value, ignoring case
    groups = (df.where(F.col(c).isNotNull())
                .groupBy(F.lower(F.trim(F.col(c))).alias("norm"), F.trim(F.col(c)).alias("spelling")).count()
                .groupBy("norm").agg(F.sum("count").alias("total"), F.max("count").alias("top"),
                                     F.count("*").alias("n_spellings"))
                .where("n_spellings > 1"))
    return int(groups.agg(F.sum(F.col("total") - F.col("top"))).first()[0] or 0)

def profile(df):
    # Run every data-quality check on a DataFrame whose columns are read as strings
    d = df.select(COLS)
    distinct = d.distinct()
    reused_ids = distinct.groupBy("order_id").count().where("count > 1")
    dates = d.groupBy(DATE_STATUS.alias("s")).count().collect()
    dates = {r["s"]: r["count"] for r in dates}
    checks = [("total rows", d.count()),
              ("exact duplicate rows", d.count() - distinct.count()),
              ("distinct records sharing an order_id",
               int(reused_ids.agg(F.sum("count")).first()[0] or 0))]
    checks += [(f"missing: {c}", v) for c, v in missing_counts(d, COLS).items()]
    checks += [(f"leading/trailing spaces: {c}", v) for c, v in whitespace_counts(d, TEXT_COLS).items()]
    checks += [(f"capitalization variants: {c}", case_variant_rows(d, c)) for c in TEXT_COLS]
    checks += [(f"invalid {c} (not {RULES[c]})", d.where(INVALID[c]).count()) for c in NUM_COLS]
    checks += [(f"order_date {s}", dates.get(s, 0))
               for s in ["valid, non-standard format", "invalid calendar date", "not a date", "future date"]]
    checks += [("'Unknown' placeholder values",
                d.select(sum(F.when(F.col(c) == "Unknown", 1).otherwise(0) for c in TEXT_COLS).alias("u"))
                 .agg(F.sum("u")).first()[0] or 0)]
    return checks

print("Rows:", TOTAL_ROWS, "| Columns:", len(COLS))

Rows: 1000 | Columns: 13


### 4.2 Duplicates

In [6]:
exact_dups = raw_df.groupBy(COLS).count().where("count > 1")
print("Exact duplicate rows (extra copies):", TOTAL_ROWS - raw_df.distinct().count())
exact_dups.select("order_id", "customer_name", "product", "order_date", "count").orderBy("order_id").show(truncate=False)

# The same order_id used by records that are NOT identical (different orders sharing an id)
reused = raw_df.distinct().groupBy("order_id").count().where("count > 1").select("order_id")
print("order_id values shared by different records:", reused.count())
(raw_df.distinct().join(reused, "order_id")
       .select("order_id", "customer_id", "customer_name", "product", "quantity", "order_date")
       .orderBy("order_id", "customer_id").show(truncate=False))

Exact duplicate rows (extra copies): 10
+--------+----------------+-----------------+----------+-----+
|order_id|customer_name   |product          |order_date|count|
+--------+----------------+-----------------+----------+-----+
|100055  |Pranav Iyer     |Dry Fruits Pack  |2026-08-20|2    |
|100145  |Sai Mishra      |Face Wash        |2025-08-28|2    |
|100297  |Manish Pillai   |Dry Fruits Pack  |2025-07-29|2    |
|100317  |Siddharth Mishra|Kurta            |2025-02-04|2    |
|100329  |Varun Verma     |Bedside Table    |2025-04-08|2    |
|100444  |Abhinav Pillai  |Novel            |2025-09-11|2    |
|100502  |Vikram Yadav    |Induction Cooktop|2025-03-11|2    |
|100588  |Diya Tripathi   |Bookshelf        |2026-08-09|2    |
|100692  |Varun Agarwal   |Electric Kettle  |2026-08-18|2    |
|100773  |Meera Malhotra  |Study Table      |2026-02-25|2    |
+--------+----------------+-----------------+----------+-----+

order_id values shared by different records: 8
+--------+-----------+--------

### 4.3 Missing values

In [7]:
missing = missing_counts(raw_df, COLS)
spark.createDataFrame(list(missing.items()), ["column", "missing_values"]).show(len(COLS), truncate=False)

any_missing = None
for c in COLS:
    cond = F.col(c).isNull() | (F.trim(F.col(c)) == "")
    any_missing = cond if any_missing is None else (any_missing | cond)
raw_df.where(any_missing).select("order_id", "customer_id", "customer_name", "payment_method", "city", "state").show(30, truncate=False)

+----------------+--------------+
|column          |missing_values|
+----------------+--------------+
|order_id        |0             |
|customer_id     |0             |
|customer_name   |10            |
|product         |0             |
|category        |0             |
|quantity        |0             |
|unit_price      |0             |
|discount_percent|0             |
|payment_method  |8             |
|city            |8             |
|state           |0             |
|order_date      |0             |
|order_status    |0             |
+----------------+--------------+

+--------+-----------+---------------+----------------+----------+--------------+
|order_id|customer_id|customer_name  |payment_method  |city      |state         |
+--------+-----------+---------------+----------------+----------+--------------+
|100013  |C0196      |NULL           |Credit Card     |Chennai   |Tamil Nadu    |
|100051  |C0180      |Arjun Agarwal  |NULL            |Mumbai    |Maharashtra   |
|100178  |C

### 4.4 Text-format inconsistencies

Two checks on the text columns:

* values with **leading or trailing spaces**
* the same value written with **different capitalization**, for example `Beauty` and `beauty` (values are grouped by `lower(trim(value))`)

In [8]:
ws = whitespace_counts(raw_df, TEXT_COLS)
spark.createDataFrame(list(ws.items()), ["column", "values_with_extra_spaces"]).show(truncate=False)

variants = None
for c in TEXT_COLS:
    v = (raw_df.where(F.col(c).isNotNull())
               .groupBy(F.lower(F.trim(F.col(c))).alias("normalized_value"))
               .agg(F.collect_set(c).alias("spellings"), F.count("*").alias("rows"))
               .where(F.size("spellings") > 1)
               .select(F.lit(c).alias("column"), "normalized_value",
                       F.array_join(F.transform(F.array_sort("spellings"),
                                                lambda s: F.concat(F.lit("'"), s, F.lit("'"))), ", ").alias("spellings_found"),
                       "rows"))
    variants = v if variants is None else variants.unionByName(v)

print("Values that appear in more than one spelling (quotes show hidden spaces):")
variants.orderBy("column", "normalized_value").show(50, truncate=False)

+--------------+------------------------+
|column        |values_with_extra_spaces|
+--------------+------------------------+
|customer_name |3                       |
|product       |0                       |
|category      |0                       |
|payment_method|0                       |
|city          |3                       |
|state         |0                       |
|order_status  |0                       |
+--------------+------------------------+

Values that appear in more than one spelling (quotes show hidden spaces):
+-------------+----------------+---------------------------------------+----+
|column       |normalized_value|spellings_found                        |rows|
+-------------+----------------+---------------------------------------+----+
|category     |beauty          |'Beauty', 'beauty'                     |138 |
|category     |clothing        |'CLOTHING', 'Clothing'                 |109 |
|category     |electronics     |'ELECTRONICS', 'Electronics'           |1

### 4.5 Invalid numerical values

Rules: `quantity` must be a whole number ≥ 1, `unit_price` must be > 0, and `discount_percent` must be between 0 and 100.

In [9]:
raw_df.select([num(c).alias(c) for c in NUM_COLS]).summary("count", "min", "max", "mean").show()

invalid_values = None
for c in NUM_COLS:
    v = (raw_df.where(INVALID[c]).groupBy(F.col(c).alias("value")).count()
               .select(F.lit(c).alias("column"), F.lit(RULES[c]).alias("rule"), "value", "count"))
    invalid_values = v if invalid_values is None else invalid_values.unionByName(v)
invalid_values.orderBy("column", F.col("value").cast("double")).show(50, truncate=False)

for c in NUM_COLS:
    print(f"invalid {c}: {raw_df.where(INVALID[c]).count()}")

+-------+--------+----------+----------------+
|summary|quantity|unit_price|discount_percent|
+-------+--------+----------+----------------+
|  count|    1000|      1000|            1000|
|    min|    -2.0|    -500.0|            -5.0|
|    max|    10.0|   81010.0|           120.0|
|   mean|   5.355|    5702.4|          19.948|
+-------+--------+----------+----------------+

+----------------+------------------------+-----+-----+
|column          |rule                    |value|count|
+----------------+------------------------+-----+-----+
|discount_percent|number between 0 and 100|-5   |4    |
|discount_percent|number between 0 and 100|120  |4    |
|quantity        |whole number >= 1       |-2   |5    |
|quantity        |whole number >= 1       |-1   |1    |
|quantity        |whole number >= 1       |0    |6    |
|unit_price      |number > 0              |-500 |3    |
|unit_price      |number > 0              |-250 |1    |
|unit_price      |number > 0              |0    |4    |
+------

### 4.6 Inconsistent and invalid dates

Each `order_date` is reduced to a pattern (every digit becomes `9`) to show which formats occur. Each value is then parsed with the formats seen in the data.

In [10]:
(raw_df.groupBy(F.regexp_replace("order_date", r"[0-9]", "9").alias("date_pattern")).count()
       .orderBy(F.desc("count")).show(truncate=False))

dated = raw_df.select("order_id", "order_date", PARSED_DATE.alias("parsed_date"), DATE_STATUS.alias("date_status"))
dated.groupBy("date_status").count().orderBy(F.desc("count")).show(truncate=False)
dated.where(F.col("date_status") != "valid, yyyy-MM-dd").orderBy("date_status", "order_id").show(truncate=False)

valid_dates = dated.where(F.col("parsed_date").isNotNull())
valid_dates.agg(F.min("parsed_date").alias("earliest_order"), F.max("parsed_date").alias("latest_order")).show()

+------------+-----+
|date_pattern|count|
+------------+-----+
|9999-99-99  |996  |
|99-99-9999  |1    |
|not-a-date  |1    |
|99/99/9999  |1    |
|9999/99/99  |1    |
+------------+-----+

+--------------------------+-----+
|date_status               |count|
+--------------------------+-----+
|valid, yyyy-MM-dd         |994  |
|invalid calendar date     |3    |
|valid, non-standard format|2    |
|not a date                |1    |
+--------------------------+-----+

+--------+----------+-----------+--------------------------+
|order_id|order_date|parsed_date|date_status               |
+--------+----------+-----------+--------------------------+
|100097  |31/02/2026|NULL       |invalid calendar date     |
|100634  |2025-02-30|NULL       |invalid calendar date     |
|100637  |2026-13-05|NULL       |invalid calendar date     |
|100448  |not-a-date|NULL       |not a date                |
|100358  |2025/07/14|2025-07-14 |valid, non-standard format|
|100969  |09-15-2026|2026-09-15 |valid, n

### 4.7 Discrepancy summary (Bronze)

In [11]:
bronze_profile = profile(raw_df)
(spark.createDataFrame(bronze_profile, ["check", "bronze_rows"])
      .where("bronze_rows > 0 OR check = 'total rows'")
      .show(60, truncate=False))

+-------------------------------------------------------+-----------+
|check                                                  |bronze_rows|
+-------------------------------------------------------+-----------+
|total rows                                             |1000       |
|exact duplicate rows                                   |10         |
|distinct records sharing an order_id                   |16         |
|missing: customer_name                                 |10         |
|missing: payment_method                                |8          |
|missing: city                                          |8          |
|leading/trailing spaces: customer_name                 |3          |
|leading/trailing spaces: city                          |3          |
|capitalization variants: category                      |6          |
|capitalization variants: city                          |6          |
|invalid quantity (not whole number >= 1)               |12         |
|invalid unit_price 

## 5. Basic preprocessing (Silver layer)

| Step | Operation | Method |
|---|---|---|
| 1 | Exact duplicates | `dropDuplicates()` keeps one copy of each identical row. Records that only share an `order_id` are different orders, so they are **kept** and flagged. |
| 2 | Text cleanup | `trim()` and collapse repeated spaces in every column. `initcap()` on `customer_name`, `category` and `city`, where title case is the correct form. It is **not** applied to values like `UPI`, `T-Shirt` or `USB-C Hub`, where it would damage correct values. |
| 3 | Missing values | `customer_name` is filled from other orders with the same `customer_id`. `city` is filled from `state` when that state has only one city in the data. Otherwise, and for `payment_method`, the value becomes `"Unknown"`. |
| 4 | Numeric validation | Cast to numbers. A value that breaks its rule (quantity ≤ 0, price ≤ 0, discount outside 0–100) is set to `NULL` and **the row is kept**. Guessing a replacement would invent data. |
| 5 | Dates | Parse every known format into one `DateType` (`yyyy-MM-dd`). Impossible or non-date values become `NULL` and the row is kept. |
| 6 | Traceability | A `dq_issues` column lists what was wrong with each row. The original values stay unchanged in Bronze. |

Step 2 runs before Step 3 so that the lookups for missing values use clean, consistent text.

### Step 1: Remove exact duplicate rows

In [12]:
step1 = raw_df.dropDuplicates()
rows_after_dedup = step1.count()
print(f"Rows before: {TOTAL_ROWS} | after removing exact duplicates: {rows_after_dedup} "
      f"| removed: {TOTAL_ROWS - rows_after_dedup}")

Rows before: 1000 | after removing exact duplicates: 990 | removed: 10


### Step 2: Trim spaces and standardize capitalization

In [13]:
CASE_FIX_COLS = ["customer_name", "category", "city"]

step2 = step1
for c in COLS:
    step2 = step2.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
for c in CASE_FIX_COLS:
    step2 = step2.withColumn(c, F.initcap(F.col(c)))
step2 = step2.select([F.when(F.col(c) == "", None).otherwise(F.col(c)).alias(c) for c in COLS])  # blank -> NULL

for c in ["category", "city"]:
    print(c, "values after standardization:")
    step2.groupBy(c).count().orderBy(c).show(30, truncate=False)

category values after standardization:
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clothing       |108  |
|Electronics    |104  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|Sports         |138  |
+---------------+-----+

city values after standardization:
+----------+-----+
|city      |count|
+----------+-----+
|NULL      |8    |
|Bengaluru |90   |
|Chennai   |88   |
|Coimbatore|81   |
|Delhi     |83   |
|Hyderabad |97   |
|Jaipur    |83   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Vijayawada|104  |
+----------+-----+



### Step 3: Handle missing values

In [14]:
# Names: a customer_id that maps to exactly one known name
name_lookup = (step2.where(F.col("customer_name").isNotNull())
                    .groupBy("customer_id")
                    .agg(F.countDistinct("customer_name").alias("n"), F.first("customer_name").alias("known_name"))
                    .where("n = 1").drop("n"))

# Cities: a state that has exactly one city in the data
city_lookup = (step2.where(F.col("city").isNotNull())
                    .groupBy("state")
                    .agg(F.countDistinct("city").alias("n"), F.first("city").alias("state_city"))
                    .where("n = 1").drop("n"))
print("States with a single city (used to fill a missing city):")
city_lookup.orderBy("state").show(truncate=False)

OTHER_TEXT = [c for c in TEXT_COLS if c not in ("customer_name", "city", "payment_method")]
step3 = (step2.join(name_lookup, "customer_id", "left")
              .join(city_lookup, "state", "left")
              .withColumn("_missing_name", F.col("customer_name").isNull())
              .withColumn("_missing_city", F.col("city").isNull())
              .withColumn("_missing_payment", F.col("payment_method").isNull())
              .withColumn("customer_name", F.coalesce("customer_name", "known_name", F.lit("Unknown")))
              .withColumn("city", F.coalesce("city", "state_city", F.lit("Unknown")))
              .withColumn("payment_method", F.coalesce("payment_method", F.lit("Unknown")))
              .fillna("Unknown", subset=OTHER_TEXT)
              .drop("known_name", "state_city"))

(step3.where("_missing_name OR _missing_city OR _missing_payment")
      .select("order_id", "customer_id", "customer_name", "state", "city", "payment_method")
      .orderBy("order_id").show(30, truncate=False))

States with a single city (used to fill a missing city):
+--------------+----------+
|state         |state_city|
+--------------+----------+
|Andhra Pradesh|Vijayawada|
|Delhi         |Delhi     |
|Karnataka     |Bengaluru |
|Rajasthan     |Jaipur    |
|Telangana     |Hyderabad |
|Uttar Pradesh |Lucknow   |
|West Bengal   |Kolkata   |
+--------------+----------+

+--------+-----------+---------------+--------------+----------+----------------+
|order_id|customer_id|customer_name  |state         |city      |payment_method  |
+--------+-----------+---------------+--------------+----------+----------------+
|100013  |C0196      |Divya Gupta    |Tamil Nadu    |Chennai   |Credit Card     |
|100051  |C0180      |Arjun Agarwal  |Maharashtra   |Mumbai    |Unknown         |
|100178  |C0078      |Aditi Patel    |West Bengal   |Kolkata   |Unknown         |
|100236  |C0220      |Vivaan Pillai  |Karnataka     |Bengaluru |Unknown         |
|100266  |C0181      |Shreya Gupta   |Telangana     |Hyderab

### Step 4: Validate quantity, unit price and discount percentage

In [15]:
NUM_TYPES = {"quantity": "int", "unit_price": "double", "discount_percent": "double"}

step4 = step3
for c in NUM_COLS:                       # flag first, while the columns are still strings
    step4 = step4.withColumn(f"_invalid_{c}", INVALID[c])
for c in NUM_COLS:
    step4 = step4.withColumn(c, F.when(F.col(f"_invalid_{c}"), None).otherwise(F.col(c).cast(NUM_TYPES[c])))

(step4.where(" OR ".join(f"_invalid_{c}" for c in NUM_COLS))
      .select("order_id", "product", *NUM_COLS).orderBy("order_id").show(40, truncate=False))

+--------+----------------------+--------+----------+----------------+
|order_id|product               |quantity|unit_price|discount_percent|
+--------+----------------------+--------+----------+----------------+
|100034  |Yoga Mat              |7       |1720.0    |NULL            |
|100102  |Atta 10kg             |7       |NULL      |40.0            |
|100114  |Ceiling Fan           |NULL    |2920.0    |15.0            |
|100140  |Cooking Oil 5L        |4       |790.0     |NULL            |
|100143  |Football              |NULL    |1030.0    |15.0            |
|100170  |Competitive Exam Guide|6       |NULL      |0.0             |
|100198  |Hoodie                |NULL    |2160.0    |12.0            |
|100217  |Football              |1       |650.0     |NULL            |
|100235  |Power Bank            |7       |NULL      |0.0             |
|100284  |Study Table           |2       |NULL      |30.0            |
|100310  |Vacuum Cleaner        |9       |9280.0    |NULL            |
|10036

### Step 5: Convert dates to one format

In [16]:
step5 = (step4.withColumn("_date_status", DATE_STATUS)
              .withColumn("_invalid_date", F.col("_date_status").isin("invalid calendar date", "not a date", "future date"))
              .withColumn("_raw_date", F.col("order_date"))
              .withColumn("order_date", F.when(F.col("_invalid_date"), None).otherwise(PARSED_DATE)))

(step5.where(F.col("_date_status") != "valid, yyyy-MM-dd")
      .select("order_id", F.col("_raw_date").alias("bronze_value"), "order_date", F.col("_date_status").alias("status"))
      .orderBy("status", "order_id").show(truncate=False))

+--------+------------+----------+--------------------------+
|order_id|bronze_value|order_date|status                    |
+--------+------------+----------+--------------------------+
|100097  |31/02/2026  |NULL      |invalid calendar date     |
|100634  |2025-02-30  |NULL      |invalid calendar date     |
|100637  |2026-13-05  |NULL      |invalid calendar date     |
|100448  |not-a-date  |NULL      |not a date                |
|100358  |2025/07/14  |2025-07-14|valid, non-standard format|
|100969  |09-15-2026  |2026-09-15|valid, non-standard format|
+--------+------------+----------+--------------------------+



### Step 6: Data-quality flags and final Silver DataFrame

In [17]:
shared_ids = step5.groupBy("order_id").count().where("count > 1").select("order_id", F.lit(True).alias("_shared_id"))

FLAGS = [("_shared_id", "duplicate_order_id"),
         ("_missing_name", "missing_customer_name"),
         ("_missing_city", "missing_city"),
         ("_missing_payment", "missing_payment_method"),
         ("_invalid_quantity", "invalid_quantity"),
         ("_invalid_unit_price", "invalid_unit_price"),
         ("_invalid_discount_percent", "invalid_discount_percent"),
         ("_invalid_date", "invalid_order_date")]

step6 = step5.join(shared_ids, "order_id", "left").fillna(False, subset=["_shared_id"])
issues = F.concat_ws("; ", *[F.when(F.col(flag), F.lit(label)) for flag, label in FLAGS])

silver_df = (step6.withColumn("dq_issues", F.when(issues == "", None).otherwise(issues))
                  .select(*COLS, "dq_issues")
                  .orderBy("order_id", "customer_id")
                  .cache())

silver_df.printSchema()
silver_df.show(10, truncate=False)
print("Silver rows:", silver_df.count())
print("Rows with at least one data-quality flag:", silver_df.where(F.col("dq_issues").isNotNull()).count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)
 |-- dq_issues: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+---------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_issues|
+--------+-----------+-------------

## 6. Save to the Silver layer

* `data_lake/silver/preprocessed_sales/`: Parquet. It keeps the column types (`int`, `double`, `date`).
* `data_lake/silver/sales_silver.csv`: the same data as a single CSV file. Spark writes CSV as a folder of `part-*` files, so the single part file is moved out and renamed.

In [18]:
silver_df.coalesce(1).write.mode("overwrite").parquet(str(SILVER_TABLE))

tmp_csv = SILVER_DIR / "_tmp_csv"
silver_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(tmp_csv))
shutil.move(str(next(tmp_csv.glob("part-*.csv"))), SILVER_CSV)
shutil.rmtree(tmp_csv)

for crc in LAKE.rglob("*.crc"):          # Hadoop checksum side files are not needed
    crc.unlink()

show_tree(LAKE)

data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    |-- preprocessed_sales/
    |   |-- _SUCCESS
    |   `-- part-00000-bf0a152d-b199-4ed2-b7e0-c5f283e6c197-c000.snappy.parquet
    `-- sales_silver.csv


## 7. Verification

### 7.1 Row count before and after preprocessing

In [19]:
silver_read = spark.read.parquet(str(SILVER_TABLE))
silver_csv  = spark.read.csv(str(SILVER_CSV), header=True, inferSchema=False)   # strings, same as Bronze

counts = [("Bronze (raw)", TOTAL_ROWS),
          ("after removing exact duplicates", rows_after_dedup),
          ("Silver Parquet (read back)", silver_read.count()),
          ("Silver CSV (read back)", silver_csv.count())]
spark.createDataFrame(counts, ["stage", "rows"]).show(truncate=False)
print("Rows removed:", TOTAL_ROWS - silver_read.count(), "(exact duplicates only, every other record is preserved)")

+-------------------------------+----+
|stage                          |rows|
+-------------------------------+----+
|Bronze (raw)                   |1000|
|after removing exact duplicates|990 |
|Silver Parquet (read back)     |990 |
|Silver CSV (read back)         |990 |
+-------------------------------+----+

Rows removed: 10 (exact duplicates only, every other record is preserved)


### 7.2 Read the Silver data back with Spark

In [20]:
silver_read.printSchema()
silver_read.show(10, truncate=False)

print("Rows that were corrected or flagged:")
silver_read.where(F.col("dq_issues").isNotNull()).show(10, truncate=False)

(silver_read.select(F.explode(F.split("dq_issues", "; ")).alias("issue"))
            .groupBy("issue").count().orderBy(F.desc("count")).show(truncate=False))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_issues: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+---------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_issues|
+--------+-----------+---------------+----

### 7.3 Same checks: Bronze vs Silver

The Section 4 checks run again on the Silver CSV, read as strings in the same way as Bronze. The remaining `missing:` counts for the numeric and date columns are the invalid values that were deliberately set to `NULL`.

In [21]:
silver_profile = dict(profile(silver_csv))
comparison = [(check, int(b), int(silver_profile[check])) for check, b in bronze_profile]
(spark.createDataFrame(comparison, ["check", "bronze", "silver"])
      .where("bronze > 0 OR silver > 0")
      .show(60, truncate=False))

print("Bronze file unchanged:", sha256(BRONZE_FILE) == BRONZE_SHA256)

+-------------------------------------------------------+------+------+
|check                                                  |bronze|silver|
+-------------------------------------------------------+------+------+
|total rows                                             |1000  |990   |
|exact duplicate rows                                   |10    |0     |
|distinct records sharing an order_id                   |16    |16    |
|missing: customer_name                                 |10    |0     |
|missing: quantity                                      |0     |12    |
|missing: unit_price                                    |0     |8     |
|missing: discount_percent                              |0     |8     |
|missing: payment_method                                |8     |0     |
|missing: city                                          |8     |0     |
|missing: order_date                                    |0     |4     |
|leading/trailing spaces: customer_name                 |3     |

### 7.4 Preprocessing performed

1. **Bronze kept as-is.** The raw `sales.csv` is a byte-for-byte, read-only copy, confirmed by its SHA-256 checksum before and after processing.
2. **Exact duplicate rows removed** with `dropDuplicates()`. Different orders that share an `order_id` were kept and flagged `duplicate_order_id`.
3. **Text standardized.** Spaces were trimmed in every column. `customer_name`, `category` and `city` were converted to title case, which merges variants such as `ELECTRONICS`/`Electronics` and `DELHI`/`delhi`/`Delhi`.
4. **Missing values handled.** `customer_name` was recovered from the same `customer_id`. `city` was inferred from `state` where the state has only one city. Anything else, and `payment_method`, was set to `"Unknown"`.
5. **Numbers validated.** `quantity` (whole number ≥ 1), `unit_price` (> 0) and `discount_percent` (0–100) were cast to numeric types. Values that broke a rule were set to `NULL` and the rows were kept.
6. **Dates standardized.** `yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy` and `MM-dd-yyyy` were converted to one `DateType` (`yyyy-MM-dd`). Impossible dates and non-dates became `NULL`.
7. **Traceability.** Each row's `dq_issues` column records what was corrected, so later layers can filter or review those rows.
8. **Saved to Silver** as Parquet (`preprocessed_sales/`) and as a single CSV (`sales_silver.csv`), then read back with Spark.

The goal was a usable Silver layer, not a perfectly clean dataset. No rows were dropped apart from exact duplicates, and unusual but plausible values were left alone.

## 8. (Colab only) Download the outputs

Downloads a zip of the whole `data_lake/` folder. It contains `silver/sales_silver.csv` for the submission.

In [22]:
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("data_lake", "zip", ".", "data_lake")
    files.download("data_lake.zip")
else:
    print("Not running in Colab: outputs are in", LAKE.resolve())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
spark.stop()